In [1]:
# pip install mygene

In [98]:
import pandas as pd
import numpy as np
import requests
import time
import mygene

In [99]:
df = pd.read_csv("datosGene4PD/t_common_variant.txt", sep = "\t", index_col = False)

In [100]:
df.head()

,Chr,gene_symbol,SNPs_symbol,SNP_position,effect_allele,alternate_allele,joint_phase_P,joint_phase_OR,joint_phase_OR_CI,pubMed_ID,Unnamed: 10
0,6,GPR126,rs757765789,142758601,T,G,1.42E-06,1.06,1.016-1.098,28256260,NaN
1,1,SYT11,rs202015799,155839054,C,T,4.70E-09,-,-,24842889,NaN
2,12,SLC2A13,rs1994090,40428561,G,T,3.20E-54,12.05,8.35-17.41,24842889,NaN
3,12,SLC2A13,rs2708453,40478652,G,T,3.62E-54,12.05,8.35-17.41,24842889,NaN
4,12,SLC2A13,rs4768212,40474147,C,T,3.62E-54,12.05,8.35-17.41,24842889,NaN


In [71]:
# df_genes = df["gene_symbol"]

In [72]:
# df_genes

In [73]:
# df_genes_t = df_genes.dropna()

In [74]:
# df_genes_t = df_genes_t.reset_index(drop=False)["gene_symbol"]

In [75]:
# df_genes_t

In [76]:
# df_genes_t.head(20)

In [101]:
def extrae_gene_symbols(dataframe):

    df_corregido = df.drop('Unnamed: 10', axis = 1)
    df_corregido = df_corregido.dropna(subset = ['gene_symbol'])
    df_corregido = df_corregido.reset_index(drop=False)
    df_corregido = df_corregido.drop('index', axis = 1)
    df_genes = df_corregido["gene_symbol"]
    
    lista_symbols = []
    
    for i, gene in enumerate(df_genes):
        
        gene = gene.replace(",", ";")
    
        if "dist" in gene:
            continue

        elif ";" in gene:

            separacion1 = gene.split(";")

            for gen in separacion1:
                lista_symbols.append(gen)

        else:
            lista_symbols.append(gene)

    lista_unicos = []
    
    for symbol in lista_symbols:
        
        if symbol not in lista_unicos:
            
            lista_unicos.append(symbol)
                
    return df_corregido, lista_symbols, lista_unicos

In [102]:
df_corregido, lista_symbols, lista_unicos = extrae_gene_symbols(df)

In [106]:
def extrae_SNPs(df_corregido):
    
    df_corregido["gene_symbol"] = df_corregido["gene_symbol"].str.split(r'\s*[;,]\s*')

    df_corregido = df_corregido.explode("gene_symbol").reset_index(drop = True)

    df_SNPs = df_corregido[["Chr", "gene_symbol", "SNP_position", "effect_allele", "alternate_allele"]]

    df_SNPs = df_SNPs[~df_SNPs['Chr'].str.contains('rs', na = False)]

    return df_SNPs

In [107]:
df_SNPs = extrae_SNPs(df_corregido)

In [109]:
def extrae_coords_inicio_fin(lista_unicos):

    mg = mygene.MyGeneInfo()
    
    resultados_coords = mg.querymany(lista_unicos, scopes = "symbol,alias", fields = "genomic_pos_hg19", species = "human")

    coords_genes = []
    no_encontrados = []
    
    for resultado in resultados_coords:
        
        gen = resultado.get("query")
    
        if resultado.get("notfound"):
            no_encontrados.append(gen)
            continue
    
        posicion = resultado.get("genomic_pos_hg19")
        
        if isinstance(posicion, list):
            posicion = posicion[0]
    
        if posicion:
            coords_genes.append({"gene_symbol": gen, "chr": str(posicion.get("chr")), "inicio": posicion.get("start"), "fin": posicion.get("end"), "cadena": posicion.get("strand")})

    df_coords = pd.DataFrame(coords_genes)

    return df_coords, no_encontrados

In [110]:
df_coords, no_encontrados = extrae_coords_inicio_fin(lista_unicos)

43 input query terms found dup hits:	[('TBC1D3P2', 2), ('CAST', 4), ('DHFRP3', 2), ('HLA-DQB1', 2), ('BST1', 2), ('CASC6', 2), ('PWRN4', 
48 input query terms found no hit:	['SLC2A15', 'NONE', 'LOC440311', '43160', 'LOC100133091', 'LOC101928978', 'MIR7641-2', 'LOC201175', 


In [113]:
dicc = {}
for gen in lista_unicos:
    dicc.update({gen: {'Chr':0, 'Inicio':0, 'SNPs': [], 'Fin':0, 'Cadena':0}})

In [114]:
for i in range(len(df_SNPs)):
    
    gen = df_SNPs.iloc[i]['gene_symbol']
    
    if lista_symbols.count(gen) == 1:
        
        dicc[gen]['Chr'] = df_SNPs.iloc[i]['Chr']
        dicc[gen]['SNPs'] = df_SNPs.iloc[i]['SNP_position']
        
    elif lista_symbols.count(gen) > 1:
            
        dicc[gen]['Chr'] = df_SNPs.iloc[i]['Chr']
        dicc[gen]['SNPs'].append(df_SNPs.iloc[i]['SNP_position'])
        
    else:
        continue

In [117]:
len(df_SNPs)

1358

In [118]:
len(df_coords)

599

In [121]:
df_completo = pd.merge(df_SNPs, df_coords, on = "gene_symbol", how = "left")

In [122]:
df_completo

,Chr,gene_symbol,SNP_position,effect_allele,alternate_allele,chr,inicio,fin,cadena
0,6,GPR126,142758601,T,G,6,142622991.0,142767403.0,1.0
1,1,SYT11,155839054,C,T,1,155829300.0,155854990.0,1.0
2,12,SLC2A13,40428561,G,T,12,40148823.0,40499891.0,-1.0
3,12,SLC2A13,40478652,G,T,12,40148823.0,40499891.0,-1.0
4,12,SLC2A13,40474147,C,T,12,40148823.0,40499891.0,-1.0
...,...,...,...,...,...,...,...,...,...
1401,17,DNAH17,76425480,A,T,17,76419778.0,76573476.0,-1.0
1402,18,ASXL3,31304318,T,G,18,31158579.0,31331156.0,1.0
1403,18,MEX3C,48683589,T,G,18,48700920.0,48744674.0,-1.0
1404,20,CRLS1,6006041,T,C,20,5986736.0,6020699.0,1.0


In [12]:
# for gen in lista_symbols:
#     print(gen)

In [89]:
# mg = mygene.MyGeneInfo()

In [77]:
# gen = ["GPR126"]
# resultado_prueba = mg.querymany(gen, scopes = "symbol,alias", fields = "genomic_pos_hg19", species = "human")

In [78]:
# print(resultado_prueba)

In [87]:
# lista_unicos = []
# for symbol in lista_symbols:
#     if symbol not in lista_unicos:
#         lista_unicos.append(symbol)

In [90]:
# resultados_coords = mg.querymany(lista_unicos, scopes = "symbol,alias", fields = "genomic_pos_hg19", species = "human")

In [24]:
# print(resultados_coords)

In [11]:
# coords_genes = []
# no_encontrados = []

# for resultado in resultados_coords:
    
#     gen = resultado.get("query")

#     if resultado.get("notfound"):
#         no_encontrados.append(gen)
#         continue

#     posicion = resultado.get("genomic_pos_hg19")
    
#     if isinstance(posicion, list):
#         posicion = posicion[0]

#     if posicion:
#         coords_genes.append({"gene_symbol": gen, "chr": str(posicion.get("chr")), "inicio": posicion.get("start"), "fin": posicion.get("end"), "cadena": posicion.get("strand")})



In [36]:
# no_encontrados

In [38]:
# coords_genes

In [12]:
# df_coords = pd.DataFrame(coords_genes)

In [91]:
# df_coords

In [79]:
# df['Unnamed: 10'].isna().all()

In [92]:
# df_corregido

In [80]:
# df_SNPs = df_corregido[["Chr", "gene_symbol", "SNP_position", "effect_allele", "alternate_allele"]]

In [81]:
# df_SNPs

In [82]:
# df_SNPs = df_SNPs[~df_SNPs['Chr'].str.contains('rs', na = False)]

In [83]:
# df_SNPs

In [19]:
# for i in range(len(df_SNPs)):
#     if 
#     loc_genes = {df_SNPs.iloc[i]['gene_symbol']: {'Chr': df_SNPs.iloc[i]['Chr'], 'SNP_pos': df_SNPs.iloc[i]['SNP_position']}}
#     break

In [20]:
# loc_genes

In [27]:
# dicc = {}
# for gen in lista_unicos:
#     dicc.update({gen: {'Chr':0, 'Inicio':0, 'SNPs': [], 'Fin':0, 'Cadena':0}})

In [28]:
# for i in range(len(df_SNPs)):
    
#     gen = df_SNPs.iloc[i]['gene_symbol']
    
#     if lista_symbols.count(gen) == 1:
        
#         dicc[gen]['Chr'] = df_SNPs.iloc[i]['Chr']
#         dicc[gen]['SNPs'] = df_SNPs.iloc[i]['SNP_position']
        
#     elif lista_symbols.count(gen) > 1:
            
#         dicc[gen]['Chr'] = df_SNPs.iloc[i]['Chr']
#         dicc[gen]['SNPs'].append(df_SNPs.iloc[i]['SNP_position'])
        
#     else:
#         continue

In [93]:
# df_corregido["gene_symbol"] = df_corregido["gene_symbol"].str.split(r'\s*[;,]\s*')

In [94]:
# df_corregido.head(20)

In [95]:
# df_corregido = df_corregido.explode("gene_symbol").reset_index(drop = True)

In [96]:
# df_corregido

In [97]:
# df_corregido.head(20)